In [1]:
import pandas as pd

In [2]:
train_df = pd.read_csv('../data/train.csv')
pred_df = pd.read_csv('../predictions-textmeta/train_predictions_xgboost_reg.csv')

In [3]:
analysis_df = pd.merge(train_df, pred_df, on='id', how='inner')
analysis_df = analysis_df.drop(columns=['actual_intensity', 'actual_emotional_state'])
analysis_df.head()

,id,journal_text,ambience_type,duration_min,sleep_hours,energy_level,stress_level,time_of_day,previous_day_mood,face_emotion_hint,reflection_quality,emotional_state,intensity,predicted_emotional_state,predicted_intensity,confidence_score,uncertainty_flag
0,1,The ocean ambience helped me stop drifting and...,ocean,12,6.5,4,2,afternoon,mixed,calm_face,clear,focused,3,focused,3,0.971016,False
1,2,"I tried to relax during the forest ambience, y...",forest,35,6.0,2,4,evening,calm,tired_face,vague,restless,3,restless,3,0.967917,False
2,3,The forest session slowed my thoughts and I fe...,forest,3,NaN,2,1,night,overwhelmed,happy_face,clear,calm,3,calm,3,0.998236,False
3,4,"the mountain ambience was pleasant, though i c...",mountain,25,7.0,4,4,night,focused,calm_face,vague,neutral,1,neutral,2,0.877595,False
4,5,"The rain session gave me a pause, but the pres...",rain,25,5.0,3,5,afternoon,NaN,tense_face,clear,overwhelmed,5,overwhelmed,4,0.986388,False


In [4]:
#check clashes where the model is making wrong predictions or flagged as uncertain (make bollean masks and filter the dataframe)
state_error_mask = analysis_df['emotional_state'] != analysis_df['predicted_emotional_state']
intensity_error_mask = analysis_df['intensity'] != analysis_df['predicted_intensity']
uncertainty_mask = analysis_df['uncertainty_flag'] == True

problem_df = analysis_df[state_error_mask | intensity_error_mask | uncertainty_mask]
print(len(problem_df))
problem_df.head()

704


,id,journal_text,ambience_type,duration_min,sleep_hours,energy_level,stress_level,time_of_day,previous_day_mood,face_emotion_hint,reflection_quality,emotional_state,intensity,predicted_emotional_state,predicted_intensity,confidence_score,uncertainty_flag
3,4,"the mountain ambience was pleasant, though i c...",mountain,25,7.0,4,4,night,focused,calm_face,vague,neutral,1,neutral,2,0.877595,False
4,5,"The rain session gave me a pause, but the pres...",rain,25,5.0,3,5,afternoon,NaN,tense_face,clear,overwhelmed,5,overwhelmed,4,0.986388,False
11,12,I feel mentally clear after the mountain sessi...,mountain,12,5.5,3,2,morning,restless,neutral_face,clear,focused,4,focused,3,0.883091,False
12,13,"The forest session made me calmer, but part of...",forest,20,5.0,2,2,afternoon,neutral,none,conflicted,mixed,2,mixed,3,0.990482,False
14,15,"I feel lighter after the mountain sounds, like...",mountain,8,8.0,3,3,night,overwhelmed,calm_face,clear,calm,2,calm,3,0.992753,False


In [5]:
def flag_error_type(row):
    errors = []
    if row["emotional_state"] != row["predicted_emotional_state"]:
        errors.append("State")
    if row["intensity"] != row["predicted_intensity"]:
        errors.append("Intensity")
    if row["uncertainty_flag"]:
        errors.append("Uncertain")
    return " & ".join(errors)
problem_df["error_type"] = problem_df.apply(flag_error_type, axis=1)
problem_df.head()

,id,journal_text,ambience_type,duration_min,sleep_hours,energy_level,stress_level,time_of_day,previous_day_mood,face_emotion_hint,reflection_quality,emotional_state,intensity,predicted_emotional_state,predicted_intensity,confidence_score,uncertainty_flag,error_type
3,4,"the mountain ambience was pleasant, though i c...",mountain,25,7.0,4,4,night,focused,calm_face,vague,neutral,1,neutral,2,0.877595,False,Intensity
4,5,"The rain session gave me a pause, but the pres...",rain,25,5.0,3,5,afternoon,NaN,tense_face,clear,overwhelmed,5,overwhelmed,4,0.986388,False,Intensity
11,12,I feel mentally clear after the mountain sessi...,mountain,12,5.5,3,2,morning,restless,neutral_face,clear,focused,4,focused,3,0.883091,False,Intensity
12,13,"The forest session made me calmer, but part of...",forest,20,5.0,2,2,afternoon,neutral,none,conflicted,mixed,2,mixed,3,0.990482,False,Intensity
14,15,"I feel lighter after the mountain sounds, like...",mountain,8,8.0,3,3,night,overwhelmed,calm_face,clear,calm,2,calm,3,0.992753,False,Intensity


In [6]:
problem_df["error_type"].value_counts()

error_type
Intensity                        467
State & Intensity                146
State                             39
Intensity & Uncertain             21
Uncertain                         15
State & Intensity & Uncertain     12
State & Uncertain                  4
Name: count, dtype: int64

In [7]:
#errors in intesity are more common... and i currenlty dont have a way to see confidence scores for them since they were treated as regression problem and not classification probelm. So I will test it as classification model for intesity as well.